# MODULE 27 — Hugging Face Transformers & Fine-Tuning

**Project:** Fine-tune DistilBERT for sentiment classification

### Learning flow

```text
Core APIs
   ↓
Dataset Pipeline
   ↓
Training
   ↓
Fine-Tuning Workflow
   ↓
Sentiment Classification Project
   ↓
Inference
```

> Run the notebook **cell by cell** in order.


## 28.1 Core APIs

In [1]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

import evaluate


In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)


In [ ]:
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print(tokenizer)
print("Vocabulary size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)


In [ ]:
text = "I really enjoyed this movie."

tokens = tokenizer(text)

print(tokens)


In [ ]:
print("Input IDs:")
print(tokens["input_ids"])

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"]))


In [ ]:
texts = [
    "I loved this movie.",
    "This movie was terrible.",
    "The acting was excellent."
]

batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)


In [ ]:
base_model = AutoModel.from_pretrained(model_name)

base_model.to(device)
base_model.eval()

print(type(base_model))


In [ ]:
inputs = {
    key: value.to(device)
    for key, value in batch.items()
}

with torch.no_grad():
    outputs = base_model(**inputs)

print("Last hidden state shape:")
print(outputs.last_hidden_state.shape)


In [ ]:
sequence_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print(type(sequence_model))


In [ ]:
sequence_model.to(device)
sequence_model.eval()

with torch.no_grad():
    outputs = sequence_model(**inputs)

print("Logits shape:", outputs.logits.shape)
print("Logits:")
print(outputs.logits)


In [ ]:
probabilities = torch.softmax(outputs.logits, dim=-1)

print("Probabilities:")
print(probabilities)

print("\nPredicted class:")
print(torch.argmax(probabilities, dim=-1))


In [ ]:
token_model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3
)

print(type(token_model))


In [ ]:
token_model.to(device)
token_model.eval()

with torch.no_grad():
    token_outputs = token_model(**inputs)

print("Logits shape:")
print(token_outputs.logits.shape)


In [ ]:
qa_model = AutoModelForQuestionAnswering.from_pretrained(
    model_name
)

print(type(qa_model))


In [ ]:
question = "What did I enjoy?"
context = "I really enjoyed this movie."

qa_inputs = tokenizer(
    question,
    context,
    return_tensors="pt",
    truncation=True
)

qa_inputs = {
    key: value.to(device)
    for key, value in qa_inputs.items()
}

qa_model.to(device)
qa_model.eval()

with torch.no_grad():
    qa_outputs = qa_model(**qa_inputs)

print("Start logits shape:", qa_outputs.start_logits.shape)
print("End logits shape:", qa_outputs.end_logits.shape)


In [ ]:
seq2seq_model_name = "google-t5/t5-small"

seq2seq_tokenizer = AutoTokenizer.from_pretrained(
    seq2seq_model_name
)

seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(
    seq2seq_model_name
)

print(type(seq2seq_model))


In [ ]:
text = "translate English to German: I love machine learning."

seq2seq_inputs = seq2seq_tokenizer(
    text,
    return_tensors="pt"
)

seq2seq_inputs = {
    key: value.to(device)
    for key, value in seq2seq_inputs.items()
}

seq2seq_model.to(device)
seq2seq_model.eval()

with torch.no_grad():
    generated_ids = seq2seq_model.generate(
        **seq2seq_inputs,
        max_new_tokens=30
    )

result = seq2seq_tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print(result)


## 28.2 Dataset Pipeline

We will use the **Rotten Tomatoes** sentiment dataset.

Pipeline:

```text
Dataset
  ↓
Inspect
  ↓
Tokenizer
  ↓
Batched map()
  ↓
Dynamic padding
  ↓
Data collator
```


In [ ]:
dataset = load_dataset(
    "cornell-movie-review-data/rotten_tomatoes"
)

print(dataset)


In [ ]:
print(dataset)


In [ ]:
print(dataset["train"][0])


In [ ]:
print(dataset["train"].features)


In [ ]:
for split in dataset:
    print(f"{split}: {len(dataset[split])} examples")


In [ ]:
for i in range(5):
    example = dataset["train"][i]

    print("Text:", example["text"])
    print("Label:", example["label"])
    print("-" * 80)


In [ ]:
id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

print(id2label)
print(label2id)


In [ ]:
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )


In [ ]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)


In [ ]:
print(tokenized_dataset["train"][0])


In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(data_collator)


In [ ]:
sample_features = [
    tokenized_dataset["train"][0],
    tokenized_dataset["train"][1],
    tokenized_dataset["train"][2],
]

batch = data_collator(sample_features)

print("Input IDs shape:")
print(batch["input_ids"].shape)

print("\nAttention mask shape:")
print(batch["attention_mask"].shape)

print("\nLabels:")
print(batch["labels"])


## 28.3 Training

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print(model.config)


In [ ]:
print("Model device:", next(model.parameters()).device)


In [ ]:
accuracy_metric = evaluate.load("accuracy")


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(
        predictions,
        axis=-1
    )

    return accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )


In [ ]:
test_predictions = np.array([
    [0.1, 0.9],
    [0.8, 0.2],
    [0.2, 0.8],
    [0.7, 0.3]
])

test_labels = np.array([
    1,
    0,
    1,
    0
])

print(
    compute_metrics(
        (test_predictions, test_labels)
    )
)


In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_sentiment_checkpoints",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=2,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",
    greater_is_better=True,

    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=100,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=42
)

print(training_args)


In [ ]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


In [ ]:
train_result = trainer.train()


In [ ]:
print(train_result)


In [ ]:
eval_results = trainer.evaluate()

print(eval_results)


In [ ]:
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best metric:",
    trainer.state.best_metric
)


In [ ]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"]
)

print(test_results)


## 28.4 Complete Fine-Tuning Workflow

```text
Pretrained Model
       ↓
Task Dataset
       ↓
Tokenizer
       ↓
Data Collator
       ↓
Fine-Tuning
       ↓
Validation
       ↓
Checkpoint
       ↓
Inference
```


In [ ]:
final_model_dir = "./final_sentiment_model"

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print("Saved model to:", final_model_dir)


In [ ]:
print(os.listdir(final_model_dir))


In [ ]:
loaded_tokenizer = AutoTokenizer.from_pretrained(
    final_model_dir
)

loaded_model = AutoModelForSequenceClassification.from_pretrained(
    final_model_dir
)

loaded_model.to(device)
loaded_model.eval()

print("Model successfully reloaded.")


## 28.5 Project — Sentiment Classification

In [ ]:
def predict_sentiment(text):
    inputs = loaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = loaded_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[
        0,
        predicted_id
    ].item()

    label = loaded_model.config.id2label[
        predicted_id
    ]

    return {
        "label": label,
        "confidence": confidence
    }


In [ ]:
text = "This movie was absolutely fantastic!"

result = predict_sentiment(text)

print(result)


In [ ]:
text = "This movie was painfully boring and disappointing."

result = predict_sentiment(text)

print(result)


In [ ]:
texts = [
    "I loved every minute of this movie.",
    "The movie was awful.",
    "The acting was brilliant.",
    "I would not recommend this film.",
    "One of the best movies I have seen."
]

for text in texts:
    result = predict_sentiment(text)

    print(f"Text: {text}")
    print(f"Prediction: {result['label']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print("-" * 80)


## Final End-to-End Inference

```text
Raw Text
   ↓
Tokenizer
   ↓
Input IDs + Attention Mask
   ↓
Fine-Tuned DistilBERT
   ↓
Logits
   ↓
Softmax
   ↓
Predicted Label + Confidence
```


In [ ]:
text = "The movie was surprisingly good."

inputs = loaded_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = loaded_model(**inputs)

probabilities = torch.softmax(
    outputs.logits,
    dim=-1
)

predicted_id = torch.argmax(
    probabilities,
    dim=-1
).item()

predicted_label = loaded_model.config.id2label[
    predicted_id
]

confidence = probabilities[
    0,
    predicted_id
].item()

print("Text:", text)
print("Prediction:", predicted_label)
print("Confidence:", f"{confidence:.4f}")


# Module 27 — Completion Checklist

- [x] `AutoTokenizer`
- [x] `AutoModel`
- [x] `AutoModelForSequenceClassification`
- [x] `AutoModelForTokenClassification`
- [x] `AutoModelForQuestionAnswering`
- [x] `AutoModelForSeq2SeqLM`
- [x] Dataset loading
- [x] Dataset inspection
- [x] Batched `map()`
- [x] Dynamic padding
- [x] `DataCollatorWithPadding`
- [x] `TrainingArguments`
- [x] `Trainer`
- [x] Validation
- [x] Checkpoints
- [x] Best-model loading
- [x] Test evaluation
- [x] Model saving
- [x] Model reloading
- [x] Inference
- [x] End-to-end sentiment classification project
